# Chapter 8 Companion — Interactive Supervised Learning for Risk Prediction

This notebook accompanies Chapter 8 of *Risk Analytics: Machine Learning and Optimization for Data-Driven Decision Making*.

The objective is not only to reproduce the chapter figures. The objective is to let the reader **change model settings, feature sets, review capacity, severity assumptions, and decision thresholds**, then rerun the analysis.

The workflow follows the chapter: data preparation, probability prediction, model comparison, expected-loss calculation, risk concentration, and model-driver interpretation.

## 1. Setup

Run this cell first. The notebook uses only a lightweight scientific Python stack: NumPy, pandas, matplotlib, and scikit-learn.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

ROOT = Path.cwd()
DATA_PATH = ROOT / "data" / "default of credit card clients.xls"
FIG_DIR = ROOT / "figures" / "chapter_8"
TAB_DIR = ROOT / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

## 2. User-configurable settings

Change this cell to run alternative experiments. Then rerun the cells below.

Key choices:

- `FEATURE_SET`: select which variables enter the models.
- `SEVERITY_ASSUMPTIONS`: change the assumed loss rate if default occurs.
- `REVIEW_SHARES`: change how many highest-risk accounts are selected for review.
- model hyperparameters: change tree depth, number of trees, learning rate, and so on.
- `DECISION_THRESHOLD`: change how probabilities are converted into predicted default labels.

In [ ]:
CONFIG = {
    # Data splitting
    "RANDOM_STATE": 42,
    "TRAIN_SIZE": 0.60,
    "VALIDATION_SIZE": 0.20,
    "TEST_SIZE": 0.20,

    # Choose one: "chapter", "raw_repayment", "all_numeric", "minimal"
    "FEATURE_SET": "chapter",

    # Model hyperparameters
    "TREE_MAX_DEPTH": 3,
    "LOGISTIC_C": 1.0,
    "RF_N_ESTIMATORS": 300,
    "RF_MAX_DEPTH": 6,
    "RF_MIN_SAMPLES_LEAF": 50,
    "GB_N_ESTIMATORS": 120,
    "GB_LEARNING_RATE": 0.05,
    "GB_MAX_DEPTH": 2,
    "GB_MIN_SAMPLES_LEAF": 50,

    # Risk analytics assumptions
    "BASE_SEVERITY": 0.50,
    "SEVERITY_ASSUMPTIONS": [0.30, 0.50, 0.70],
    "REVIEW_SHARES": [0.05, 0.10, 0.20],
    "DECISION_THRESHOLD": 0.50,

    # Output controls
    "SAVE_OUTPUTS": True,
}

CONFIG

## 3. Load and prepare the data

The dataset is the Taiwan credit-card default dataset. The notebook creates both original variables and chapter-style aggregate variables.

In [ ]:
def load_credit_data(path: Path) -> pd.DataFrame:
    raw = pd.read_excel(path, header=1)
    raw = raw.rename(columns={raw.columns[0]: "ID"})
    target_col = "default payment next month"
    raw = raw.rename(columns={target_col: "default"})
    return raw


def add_chapter_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    pay_status_cols = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
    bill_cols = ["BILL_AMT1", "BILL_AMT2", "BILL_AMT3", "BILL_AMT4", "BILL_AMT5", "BILL_AMT6"]
    pay_amt_cols = ["PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"]

    df["PAY_STATUS_MEAN"] = df[pay_status_cols].mean(axis=1)
    df["PAY_STATUS_MAX"] = df[pay_status_cols].max(axis=1)
    df["BILL_AMT_MEAN"] = df[bill_cols].mean(axis=1)
    df["PAY_AMT_MEAN"] = df[pay_amt_cols].mean(axis=1)
    df["UTILIZATION_MEAN"] = df["BILL_AMT_MEAN"] / df["LIMIT_BAL"].replace(0, np.nan)
    df["PAY_TO_BILL_MEAN"] = df["PAY_AMT_MEAN"] / df["BILL_AMT_MEAN"].replace(0, np.nan)
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.fillna(0)
    return df


def get_feature_columns(df: pd.DataFrame, feature_set: str):
    chapter_features = [
        "LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE",
        "PAY_0", "PAY_2", "PAY_3", "PAY_4",
        "PAY_STATUS_MEAN", "PAY_STATUS_MAX",
        "BILL_AMT_MEAN", "PAY_AMT_MEAN", "UTILIZATION_MEAN", "PAY_TO_BILL_MEAN",
    ]
    raw_repayment = [
        "LIMIT_BAL", "AGE", "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6",
        "BILL_AMT1", "BILL_AMT2", "BILL_AMT3", "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
        "PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6",
    ]
    minimal = ["LIMIT_BAL", "AGE", "PAY_0", "PAY_STATUS_MEAN", "PAY_STATUS_MAX", "UTILIZATION_MEAN"]

    if feature_set == "chapter":
        cols = chapter_features
    elif feature_set == "raw_repayment":
        cols = raw_repayment
    elif feature_set == "minimal":
        cols = minimal
    elif feature_set == "all_numeric":
        cols = [c for c in df.columns if c not in ["ID", "default"]]
    else:
        raise ValueError(f"Unknown FEATURE_SET: {feature_set}")
    return [c for c in cols if c in df.columns]


df = add_chapter_features(load_credit_data(DATA_PATH))
features = get_feature_columns(df, CONFIG["FEATURE_SET"])
target = "default"

print(f"Rows: {len(df):,}")
print(f"Default rate: {df[target].mean():.2%}")
print(f"Feature set: {CONFIG['FEATURE_SET']}")
print(f"Number of features: {len(features)}")
features

## 4. Train, validation, and test splits

The chapter uses a train-validation-test design. The test set is reserved for final reported performance.

In [ ]:
X = df[features].copy()
y = df[target].astype(int).copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    train_size=CONFIG["TRAIN_SIZE"],
    stratify=y,
    random_state=CONFIG["RANDOM_STATE"],
)

relative_val_size = CONFIG["VALIDATION_SIZE"] / (CONFIG["VALIDATION_SIZE"] + CONFIG["TEST_SIZE"])
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    train_size=relative_val_size,
    stratify=y_temp,
    random_state=CONFIG["RANDOM_STATE"],
)

print(f"Train: {len(X_train):,}, default rate {y_train.mean():.2%}")
print(f"Validation: {len(X_val):,}, default rate {y_val.mean():.2%}")
print(f"Test: {len(X_test):,}, default rate {y_test.mean():.2%}")

## 5. Fit models from the current parameter settings

This cell fits four models:

1. shallow interpretable tree;
2. logistic regression;
3. random forest;
4. gradient boosting.

Change the hyperparameters in `CONFIG`, then rerun from this point to compare alternatives.

In [ ]:
def make_models(config):
    models = {
        "Interpretable tree": DecisionTreeClassifier(
            criterion="gini",
            max_depth=config["TREE_MAX_DEPTH"],
            min_samples_leaf=100,
            random_state=config["RANDOM_STATE"],
        ),
        "Logistic regression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                C=config["LOGISTIC_C"],
                max_iter=2000,
                solver="lbfgs",
                random_state=config["RANDOM_STATE"],
            )),
        ]),
        "Random forest": RandomForestClassifier(
            n_estimators=config["RF_N_ESTIMATORS"],
            max_depth=config["RF_MAX_DEPTH"],
            min_samples_leaf=config["RF_MIN_SAMPLES_LEAF"],
            random_state=config["RANDOM_STATE"],
            n_jobs=-1,
        ),
        "Gradient boosting": GradientBoostingClassifier(
            n_estimators=config["GB_N_ESTIMATORS"],
            learning_rate=config["GB_LEARNING_RATE"],
            max_depth=config["GB_MAX_DEPTH"],
            min_samples_leaf=config["GB_MIN_SAMPLES_LEAF"],
            random_state=config["RANDOM_STATE"],
        ),
    }
    return models

models = make_models(CONFIG)
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"Fitted: {name}")

## 6. Predict probabilities and evaluate performance

This section calculates AUC, precision-recall AUC, Brier score, and log-loss on the test set.

In [ ]:
def predict_proba(model, X):
    return model.predict_proba(X)[:, 1]

probs_test = {name: predict_proba(model, X_test) for name, model in models.items()}
probs_val = {name: predict_proba(model, X_val) for name, model in models.items()}

perf_rows = []
for name, p in probs_test.items():
    perf_rows.append({
        "Model": name,
        "Mean p": p.mean(),
        "AUC": roc_auc_score(y_test, p),
        "PR-AUC": average_precision_score(y_test, p),
        "Brier": brier_score_loss(y_test, p),
        "Log-loss": log_loss(y_test, np.clip(p, 1e-6, 1-1e-6)),
    })
performance = pd.DataFrame(perf_rows).sort_values("Log-loss")
performance

In [ ]:
if CONFIG["SAVE_OUTPUTS"]:
    performance.to_csv(TAB_DIR / "interactive_predictive_performance.csv", index=False)

## 7. ROC and precision-recall curves

These figures are regenerated from the current model settings.

In [ ]:
def plot_roc_curves(y_true, probs, path=None):
    fig, ax = plt.subplots()
    for name, p in probs.items():
        fpr, tpr, _ = roc_curve(y_true, p)
        auc = roc_auc_score(y_true, p)
        ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
    ax.plot([0, 1], [0, 1], "--", label="No skill")
    ax.set_title("ROC curves for default-prediction models")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.legend(fontsize=8)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=200, bbox_inches="tight")
    return fig

fig = plot_roc_curves(y_test, probs_test, FIG_DIR / "interactive_roc_curves.png" if CONFIG["SAVE_OUTPUTS"] else None)
plt.show()

In [ ]:
def plot_pr_curves(y_true, probs, path=None):
    fig, ax = plt.subplots()
    base_rate = y_true.mean()
    for name, p in probs.items():
        precision, recall, _ = precision_recall_curve(y_true, p)
        ap = average_precision_score(y_true, p)
        ax.plot(recall, precision, label=f"{name} (AP={ap:.3f})")
    ax.axhline(base_rate, linestyle="--", label=f"Base rate={base_rate:.3f}")
    ax.set_title("Precision-recall curves for default-prediction models")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend(fontsize=8)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=200, bbox_inches="tight")
    return fig

fig = plot_pr_curves(y_test, probs_test, FIG_DIR / "interactive_precision_recall_curves.png" if CONFIG["SAVE_OUTPUTS"] else None)
plt.show()

## 8. Threshold-based classification diagnostics

The chapter focuses on probabilities, but users often ask what happens if probabilities are converted into decisions. Change `DECISION_THRESHOLD` in `CONFIG` and rerun this section.

In [ ]:
def threshold_metrics(y_true, probs, threshold):
    rows = []
    for name, p in probs.items():
        pred = (p >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
        rows.append({
            "Model": name,
            "Threshold": threshold,
            "Predicted default share": pred.mean(),
            "Precision": precision_score(y_true, pred, zero_division=0),
            "Recall": recall_score(y_true, pred, zero_division=0),
            "F1": f1_score(y_true, pred, zero_division=0),
            "TP": tp, "FP": fp, "TN": tn, "FN": fn,
        })
    return pd.DataFrame(rows)

threshold_table = threshold_metrics(y_test, probs_test, CONFIG["DECISION_THRESHOLD"])
threshold_table

In [ ]:
if CONFIG["SAVE_OUTPUTS"]:
    threshold_table.to_csv(TAB_DIR / "interactive_threshold_metrics.csv", index=False)

## 9. Expected loss under user-defined severity assumptions

This section converts probabilities into predicted expected loss:

\[
e_i = p_i g a_i.
\]

Here, exposure is proxied by credit limit and `g` is the severity assumption chosen by the user.

In [ ]:
# Align exposure with the test set index.
exposure_test = df.loc[X_test.index, "LIMIT_BAL"].astype(float)

def expected_loss_table(probs, exposure, severity):
    rows = []
    for name, p in probs.items():
        e = p * severity * exposure.values
        rows.append({
            "Model": name,
            "Severity": severity,
            "Mean loss": e.mean(),
            "Total loss (millions)": e.sum() / 1_000_000,
            "Top 10% loss share": e[np.argsort(-p)[:int(np.ceil(0.10 * len(p)))]].sum() / e.sum(),
        })
    return pd.DataFrame(rows)

loss_base = expected_loss_table(probs_test, exposure_test, CONFIG["BASE_SEVERITY"])
loss_base

In [ ]:
severity_rows = []
for g in CONFIG["SEVERITY_ASSUMPTIONS"]:
    severity_rows.extend(expected_loss_table(probs_test, exposure_test, g).to_dict("records"))
severity_table = pd.DataFrame(severity_rows)
severity_table_pivot = severity_table.pivot(index="Model", columns="Severity", values="Total loss (millions)")
severity_table_pivot

In [ ]:
if CONFIG["SAVE_OUTPUTS"]:
    loss_base.to_csv(TAB_DIR / "interactive_expected_loss_base.csv", index=False)
    severity_table.to_csv(TAB_DIR / "interactive_severity_sensitivity_long.csv", index=False)
    severity_table_pivot.to_csv(TAB_DIR / "interactive_severity_sensitivity_pivot.csv")

In [ ]:
def plot_severity_sensitivity(severity_table, path=None):
    fig, ax = plt.subplots()
    for model_name, grp in severity_table.groupby("Model"):
        grp = grp.sort_values("Severity")
        ax.plot(grp["Severity"] * 100, grp["Total loss (millions)"], marker="o", label=model_name)
    ax.set_title("Total predicted loss under alternative severity assumptions")
    ax.set_xlabel("Severity assumption, g (%)")
    ax.set_ylabel("Total predicted loss (millions)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=200, bbox_inches="tight")
    return fig

fig = plot_severity_sensitivity(severity_table, FIG_DIR / "interactive_severity_sensitivity.png" if CONFIG["SAVE_OUTPUTS"] else None)
plt.show()

## 10. Review-capacity and risk-concentration analysis

Change `REVIEW_SHARES` in `CONFIG` to test what happens when an organization reviews the highest-risk 5%, 10%, 20%, or any other share of accounts.

In [ ]:
def review_capacity_table(y_true, probs, exposure, shares, severity):
    rows = []
    y_arr = np.asarray(y_true)
    exposure_arr = np.asarray(exposure)
    observed_loss_proxy = y_arr * severity * exposure_arr
    total_defaults = y_arr.sum()
    total_observed_loss_proxy = observed_loss_proxy.sum()
    for name, p in probs.items():
        order = np.argsort(-p)
        for share in shares:
            k = int(np.ceil(share * len(p)))
            selected = order[:k]
            rows.append({
                "Model": name,
                "Selected share": share,
                "Selected accounts": k,
                "Observed default rate selected": y_arr[selected].mean(),
                "Defaults captured": y_arr[selected].sum() / total_defaults,
                "Observed default-weighted exposure captured": observed_loss_proxy[selected].sum() / total_observed_loss_proxy,
            })
    return pd.DataFrame(rows)

review_table = review_capacity_table(
    y_test, probs_test, exposure_test, CONFIG["REVIEW_SHARES"], CONFIG["BASE_SEVERITY"]
)
review_table

In [ ]:
if CONFIG["SAVE_OUTPUTS"]:
    review_table.to_csv(TAB_DIR / "interactive_review_capacity.csv", index=False)

In [ ]:
def plot_review_capture(review_table, path=None):
    fig, ax = plt.subplots()
    for model_name, grp in review_table.groupby("Model"):
        grp = grp.sort_values("Selected share")
        ax.plot(grp["Selected share"] * 100, grp["Defaults captured"] * 100, marker="o", label=model_name)
    ax.set_title("Observed defaults captured by review capacity")
    ax.set_xlabel("Highest-risk accounts selected (%)")
    ax.set_ylabel("Observed defaults captured (%)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=200, bbox_inches="tight")
    return fig

fig = plot_review_capture(review_table, FIG_DIR / "interactive_default_capture_sensitivity.png" if CONFIG["SAVE_OUTPUTS"] else None)
plt.show()

## 11. Risk-decile diagnostics for any selected model

Change `SELECTED_MODEL` to inspect deciles for another model.

In [ ]:
SELECTED_MODEL = performance.iloc[0]["Model"]  # best by log-loss by default
# SELECTED_MODEL = "Gradient boosting"
# SELECTED_MODEL = "Random forest"
# SELECTED_MODEL = "Logistic regression"

SELECTED_MODEL

In [ ]:
def decile_diagnostics(y_true, p, exposure, severity):
    temp = pd.DataFrame({
        "y": np.asarray(y_true),
        "p": np.asarray(p),
        "exposure": np.asarray(exposure),
    })
    temp["expected_loss"] = temp["p"] * severity * temp["exposure"]
    temp = temp.sort_values("p").reset_index(drop=True)
    temp["decile"] = pd.qcut(temp.index + 1, 10, labels=False) + 1
    out = temp.groupby("decile").agg(
        Accounts=("y", "size"),
        Mean_p=("p", "mean"),
        Observed_default=("y", "mean"),
        Mean_exposure=("exposure", "mean"),
        Mean_loss=("expected_loss", "mean"),
        Loss_share=("expected_loss", lambda x: x.sum() / temp["expected_loss"].sum()),
    ).reset_index()
    return out

deciles = decile_diagnostics(y_test, probs_test[SELECTED_MODEL], exposure_test, CONFIG["BASE_SEVERITY"])
deciles

In [ ]:
if CONFIG["SAVE_OUTPUTS"]:
    deciles.to_csv(TAB_DIR / f"interactive_deciles_{SELECTED_MODEL.replace(' ', '_').lower()}.csv", index=False)

In [ ]:
def plot_decile_probability(deciles, selected_model, path=None):
    fig, ax = plt.subplots()
    ax.plot(deciles["decile"], deciles["Mean_p"] * 100, marker="o", label="Mean predicted probability")
    ax.plot(deciles["decile"], deciles["Observed_default"] * 100, marker="s", label="Observed default rate")
    ax.set_title(f"Predicted and observed default rates by risk decile: {selected_model}")
    ax.set_xlabel("Risk decile (1 = lowest predicted risk, 10 = highest)")
    ax.set_ylabel("Default rate (%)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=200, bbox_inches="tight")
    return fig

fig = plot_decile_probability(deciles, SELECTED_MODEL, FIG_DIR / "interactive_decile_probability.png" if CONFIG["SAVE_OUTPUTS"] else None)
plt.show()

In [ ]:
def plot_decile_loss_share(deciles, selected_model, path=None):
    fig, ax = plt.subplots()
    ax.bar(deciles["decile"], deciles["Loss_share"] * 100)
    ax.set_title(f"Predicted expected-loss concentration by risk decile: {selected_model}")
    ax.set_xlabel("Risk decile (1 = lowest predicted risk, 10 = highest)")
    ax.set_ylabel("Predicted expected-loss share (%)")
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=200, bbox_inches="tight")
    return fig

fig = plot_decile_loss_share(deciles, SELECTED_MODEL, FIG_DIR / "interactive_decile_loss_share.png" if CONFIG["SAVE_OUTPUTS"] else None)
plt.show()

## 12. Feature importance and model drivers

This section translates model variables into business-readable labels. Change `SELECTED_MODEL` above and rerun if needed.

In [ ]:
FEATURE_LABELS = {
    "PAY_0": "Most recent\nrepayment status",
    "PAY_STATUS_MEAN": "Average recent\nrepayment status",
    "PAY_STATUS_MAX": "Worst recent\nrepayment status",
    "PAY_2": "Repayment status\n2 months earlier",
    "PAY_3": "Repayment status\n3 months earlier",
    "PAY_4": "Repayment status\n4 months earlier",
    "PAY_5": "Repayment status\n5 months earlier",
    "PAY_6": "Repayment status\n6 months earlier",
    "PAY_AMT_MEAN": "Average recent\npayment amount",
    "UTILIZATION_MEAN": "Average credit-limit\nutilization",
    "PAY_TO_BILL_MEAN": "Average payment-to-bill\nratio",
    "LIMIT_BAL": "Credit limit",
    "BILL_AMT_MEAN": "Average recent\nbill amount",
    "AGE": "Age",
    "SEX": "Sex",
    "EDUCATION": "Education",
    "MARRIAGE": "Marriage",
}

def get_feature_importance(model, feature_names):
    if hasattr(model, "feature_importances_"):
        imp = model.feature_importances_
    elif isinstance(model, Pipeline) and hasattr(model.named_steps.get("model"), "coef_"):
        imp = np.abs(model.named_steps["model"].coef_[0])
    else:
        return None
    out = pd.DataFrame({"feature": feature_names, "importance": imp})
    out["label"] = out["feature"].map(FEATURE_LABELS).fillna(out["feature"])
    return out.sort_values("importance", ascending=False)

importance_tables = {}
for name, model in models.items():
    imp = get_feature_importance(model, features)
    if imp is not None:
        importance_tables[name] = imp
        display(imp.head(10))


In [ ]:
def plot_importance(imp, model_name, top_n=8, path=None):
    plot_df = imp.head(top_n).iloc[::-1]
    fig, ax = plt.subplots(figsize=(7, 4.8))
    ax.barh(plot_df["label"], plot_df["importance"])
    ax.set_title(f"{model_name}: main risk drivers")
    ax.set_xlabel("Feature importance")
    ax.set_ylabel("")
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=200, bbox_inches="tight")
    return fig

for name in ["Random forest", "Gradient boosting"]:
    if name in importance_tables:
        fig = plot_importance(
            importance_tables[name],
            name,
            top_n=8,
            path=FIG_DIR / f"interactive_{name.lower().replace(' ', '_')}_importance.png" if CONFIG["SAVE_OUTPUTS"] else None,
        )
        plt.show()
        if CONFIG["SAVE_OUTPUTS"]:
            importance_tables[name].to_csv(TAB_DIR / f"interactive_{name.lower().replace(' ', '_')}_importance.csv", index=False)

## 13. Experiment guide

Try the following changes in `CONFIG` and rerun the notebook:

1. Change `FEATURE_SET` from `"chapter"` to `"minimal"` or `"all_numeric"`.
2. Increase `RF_MAX_DEPTH` or reduce `RF_MIN_SAMPLES_LEAF` and examine whether log-loss improves or worsens.
3. Change `GB_LEARNING_RATE` and `GB_N_ESTIMATORS` together.
4. Change `SEVERITY_ASSUMPTIONS` to `[0.10, 0.30, 0.50, 0.80]`.
5. Change `REVIEW_SHARES` to `[0.01, 0.05, 0.10, 0.25]`.
6. Change `DECISION_THRESHOLD` to see how precision and recall change.

This turns the companion into a learning tool rather than a static replication script.